In [7]:
import numpy as np

In [8]:
def calculate_distance(x1, y1, x2, y2):
    """
    Euclidean distance between two points.
    """
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def calculate_angle(ax, ay, bx, by, cx, cy):
    """
    Calculates the angle ABC (in degrees).
    Returns an angle between 0 and 180 degrees.
    """

    ba = np.array([ax - bx, ay - by])
    bc = np.array([cx - bx, cy - by])

    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)

    if norm_ba == 0 or norm_bc == 0:
        return np.nan

    cosine = np.dot(ba, bc) / (norm_ba * norm_bc)

    # Numerical safety
    cosine = np.clip(cosine, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine))

    return angle


def calculate_knee_angle(row):
    """
    Hip -> Knee -> Ankle
    """

    return calculate_angle(
        row["hip_x"], row["hip_y"],
        row["knee_x"], row["knee_y"],
        row["ankle_x"], row["ankle_y"]
    )


def calculate_hip_angle(row):
    """
    Shoulder -> Hip -> Knee
    """

    return calculate_angle(
        row["shoulder_x"], row["shoulder_y"],
        row["hip_x"], row["hip_y"],
        row["knee_x"], row["knee_y"]
    )


def calculate_torso_angle(row):
    """
    Angle of the torso relative to vertical.
    """

    dx = row["shoulder_x"] - row["hip_x"]
    dy = row["shoulder_y"] - row["hip_y"]

    angle = np.degrees(np.arctan2(abs(dx), abs(dy)))

    return angle


def calculate_body_height(row):
    """
    Approximate body height using visible body segments.
    """

    torso = calculate_distance(
        row["shoulder_x"], row["shoulder_y"],
        row["hip_x"], row["hip_y"]
    )

    thigh = calculate_distance(
        row["hip_x"], row["hip_y"],
        row["knee_x"], row["knee_y"]
    )

    shin = calculate_distance(
        row["knee_x"], row["knee_y"],
        row["ankle_x"], row["ankle_y"]
    )

    return torso + thigh + shin


def calculate_hip_height(row):
    """
    Returns hip vertical position.
    """

    return row["hip_y"]


def calculate_knee_forward(row):
    """
    Horizontal knee travel relative to the ankle.
    """

    return abs(row["knee_x"] - row["ankle_x"])


def calculate_hip_shift(row):
    """
    Horizontal hip displacement relative to ankle.

    Useful for detecting excessive forward/backward movement.
    """

    return abs(row["hip_x"] - row["ankle_x"])

In [9]:
def calculate_hip_velocity(video_df):
    """
    Calculates hip velocity in x, y and total magnitude.
    """

    video_df["hip_velocity_x"] = video_df["hip_x"].diff().fillna(0)
    video_df["hip_velocity_y"] = video_df["hip_y"].diff().fillna(0)

    video_df["hip_velocity"] = np.sqrt(
        video_df["hip_velocity_x"]**2 +
        video_df["hip_velocity_y"]**2
    )

    return video_df


def calculate_knee_velocity(video_df):
    """
    Calculates knee velocity.
    """

    video_df["knee_velocity_x"] = video_df["knee_x"].diff().fillna(0)
    video_df["knee_velocity_y"] = video_df["knee_y"].diff().fillna(0)

    video_df["knee_velocity"] = np.sqrt(
        video_df["knee_velocity_x"]**2 +
        video_df["knee_velocity_y"]**2
    )

    return video_df


def calculate_shoulder_velocity(video_df):
    """
    Calculates shoulder velocity.
    """

    video_df["shoulder_velocity_x"] = video_df["shoulder_x"].diff().fillna(0)
    video_df["shoulder_velocity_y"] = video_df["shoulder_y"].diff().fillna(0)

    video_df["shoulder_velocity"] = np.sqrt(
        video_df["shoulder_velocity_x"]**2 +
        video_df["shoulder_velocity_y"]**2
    )

    return video_df

In [10]:
def calculate_hip_acceleration(video_df):

    video_df["hip_acceleration"] = (
        video_df["hip_velocity"]
        .diff()
        .fillna(0)
    )

    return video_df


def calculate_knee_acceleration(video_df):

    video_df["knee_acceleration"] = (
        video_df["knee_velocity"]
        .diff()
        .fillna(0)
    )

    return video_df


def calculate_shoulder_acceleration(video_df):

    video_df["shoulder_acceleration"] = (
        video_df["shoulder_velocity"]
        .diff()
        .fillna(0)
    )

    return video_df

In [11]:
def calculate_hip_jerk(video_df):

    video_df["hip_jerk"] = (
        video_df["hip_acceleration"]
        .diff()
        .fillna(0)
    )

    return video_df


def calculate_knee_jerk(video_df):

    video_df["knee_jerk"] = (
        video_df["knee_acceleration"]
        .diff()
        .fillna(0)
    )

    return video_df


def calculate_shoulder_jerk(video_df):

    video_df["shoulder_jerk"] = (
        video_df["shoulder_acceleration"]
        .diff()
        .fillna(0)
    )

    return video_df

In [12]:
def find_bottom_frame(video_df):
    """
    Returns the index of the frame where the hip reaches
    its lowest position (deepest squat).
    """

    return video_df["hip_y"].idxmax()

In [13]:
def find_descent_frames(video_df):
    """
    Returns all frames from the beginning of the squat until the bottom.
    """

    bottom = find_bottom_frame(video_df)

    return video_df.loc[:bottom]

In [14]:
def find_ascent_frames(video_df):
    """
    Returns all frames after the bottom
    until the end of the squat.
    """

    bottom = find_bottom_frame(video_df)

    return video_df.loc[bottom:]

In [15]:
def find_bottom_pause(video_df, tolerance=0.005):
    """
    Estimates how many frames the athlete stayed near the bottom position.
    """

    lowest = video_df["hip_y"].max()

    near_bottom = (
        video_df["hip_y"] >= lowest - tolerance
    )

    return near_bottom.sum()